# Embedding Generation with Enhanced Feature Extractor

In this notebook, we will:

- Use a more powerful pre-trained model (ResNet-50) to extract image embeddings.
- Generate embeddings for all pre-processed images in the matches dataset.
- Save the embeddings and related data for later use.
- Clear memory after processing to manage resources efficiently.
---

## **Step 1: Import Libraries**

We begin by importing the necessary libraries.

In [1]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import re
import gc

---

## **Step 2: Set Up Device**

We set up the device to use GPU if available, otherwise CPU.

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


---

## **Step 3: Load Pre-trained Model**

We load a pre-trained ResNet-50 model and set it to evaluation mode.

In [3]:
# Load pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)
model = model.to(device)
model.eval()

/Users/mohammedkhodorfirasal-tal/Documents/Professional/Work/Fellowship - Novartis/Data/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/mohammedkhodorfirasal-tal/Documents/Professional/Work/Fellowship - Novartis/Data/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/mohammedkhodorfirasal-tal/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|████████████████████████████████████████████████

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

---

## **Step 4: Define Helper Functions**

We define functions for image transformation and embedding generation.

In [4]:
# Transformation pipeline matching the pre-processing step
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet mean
                         std=[0.229, 0.224, 0.225])    # ImageNet std
])

# Function to generate embeddings
def generate_embeddings(image_paths, model, batch_size=32):
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='Generating Embeddings'):
        batch_paths = image_paths[i:i+batch_size]
        images = []
        for path in batch_paths:
            img = Image.open(path).convert('RGB')
            img = transform(img)
            images.append(img)
        images = torch.stack(images).to(device)
        with torch.no_grad():
            outputs = model(images)
            outputs = outputs.cpu().numpy()
            # Normalize embeddings
            outputs = outputs / np.linalg.norm(outputs, axis=1, keepdims=True)
            embeddings.append(outputs)
        # Free up memory
        del images, outputs
        torch.cuda.empty_cache()
    embeddings = np.vstack(embeddings)
    return embeddings

# Function to extract base ID from filename
def extract_base_id(filename):
    match = re.match(r'(\d+)[_.]', filename)
    return match.group(1) if match else None

---

## **Step 5: Load Preprocessed Images**

We load image paths from the preprocessed images.

In [5]:
# Define preprocessed images directory
preprocessed_dir = './matches_preprocessed/'

image_files = [f for f in os.listdir(preprocessed_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
image_paths = [os.path.join(preprocessed_dir, f) for f in image_files]
base_ids = [extract_base_id(f) for f in image_files]

---

## **Step 6: Generate Embeddings**

We generate embeddings for the preprocessed images.

In [6]:
# Generate embeddings
embeddings = generate_embeddings(image_paths, model)

Generating Embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.34s/it]


---

## **Step 7: Save Embeddings and Data**

We save the embeddings and related data to disk.

In [7]:
# Save embeddings and data
np.save('matches_embeddings.npy', embeddings)
np.save('matches_image_paths.npy', image_paths)
np.save('matches_image_files.npy', image_files)
np.save('matches_base_ids.npy', base_ids)

print('Embeddings and data saved.')

Embeddings and data saved.


---

## **Step 8: Clear Memory**

We clear variables and free up memory.

In [8]:
# Clear variables and free memory
del embeddings, image_paths, image_files, base_ids, model
torch.cuda.empty_cache()
gc.collect()

0

---

## **Step 9: Conclusion**
We have successfully generated and saved embeddings using the ResNet-50 model.